In [4]:
import numpy as np
from scipy import stats
import os 
import math
import pandas as pd

from load import load_seg_to_bin

ref_genome_folder = 'ref_genome/hg19'
exclude_chr = ['chrM', 'chrY', 'chrMT']
contact_matrix = 'ContactMatrix/GSE63525_K562_combined.hic'
resolution = 50000

# load ref genome length info
chr_length = dict()
with open(ref_genome_folder + '/hg19.len', 'r') as f:
    lines = f.read().splitlines()
    for line in lines:
        chr_name , len_ = line.split('\t')[:2]
        if chr_name in exclude_chr:
            continue
        chr_length[chr_name] = int(len_)
print('chr length : \n',chr_length)

# calculate bin number in each chr
chr_bin_number = dict()
for key in chr_length.keys():
    chr_bin_number[key] = math.floor(chr_length[key] / resolution) + 1
print('chr bin number : \n', chr_bin_number)

chr length : 
 {'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430, 'chr4': 191154276, 'chr5': 180915260, 'chr6': 171115067, 'chr7': 159138663, 'chr8': 146364022, 'chr9': 141213431, 'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895, 'chr13': 115169878, 'chr14': 107349540, 'chr15': 102531392, 'chr16': 90354753, 'chr17': 81195210, 'chr18': 78077248, 'chr19': 59128983, 'chr20': 63025520, 'chr21': 48129895, 'chr22': 51304566, 'chrX': 155270560}
chr bin number : 
 {'chr1': 4986, 'chr2': 4864, 'chr3': 3961, 'chr4': 3824, 'chr5': 3619, 'chr6': 3423, 'chr7': 3183, 'chr8': 2928, 'chr9': 2825, 'chr10': 2711, 'chr11': 2701, 'chr12': 2678, 'chr13': 2304, 'chr14': 2147, 'chr15': 2051, 'chr16': 1808, 'chr17': 1624, 'chr18': 1562, 'chr19': 1183, 'chr20': 1261, 'chr21': 963, 'chr22': 1027, 'chrX': 3106}


## r between IS and residual

In [132]:
residual_dir = 'HiNTcnv_OUTPUT/K562_combined_dataForRegression/residualsPerChrom'

residual_dict = dict()
for file in os.listdir(residual_dir):
    with open(residual_dir + '/' + file, 'r') as f:
        chr_ = file.split('_')[1]
        residual_per_chr = []
        for line in f.readlines():
            residual_per_chr.append(float(line))
        residual_dict[chr_] = residual_per_chr

In [241]:
IS_path = 'FAN-C_Insulation/bed/GSE63525_K562_50k_IS_150kb.bed'

IS_dict = dict()
for key in chr_bin_number.keys():
    IS_dict[key] = np.zeros(chr_bin_number[key])

with open(IS_path, 'r') as f:
    for line in f.readlines():
        line_split = line.strip().split()
        chr_ = 'chr'+line_split[0]
        if chr_ in exclude_chr :
            continue
        bin_ = math.floor(int(line_split[1]) / resolution)
        if (line_split[4] == 'inf') or (line_split[4] == '-inf') or (line_split[4] == 'nan'):
            IS_dict[chr_][bin_] = 0
        else:
            IS_dict[chr_][bin_] = float(line_split[4])

In [242]:
# delete IS by zero lines

zero_line = dict()
for key in chr_bin_number.keys():
    zero_line[key] = list()
with open('exp2_IS_residual/zero_line.txt', 'r') as f :
    for line in f.readlines():
        line_split = line.strip().split()
        zero_line[line_split[0]].append(int(line_split[1]))

for key in chr_bin_number.keys():
    IS_dict[key] = np.delete(IS_dict[key], zero_line[key])

In [193]:
# IS value interval
for key in chr_bin_number.keys():
    print(key, f'[{min(IS_dict[key])}, {max(IS_dict[key])}]')

chr1 [-5.497572898864746, 2.3449840545654297]
chr2 [-3.386786937713623, 2.196972608566284]
chr3 [-1.5183751583099365, 1.5183136463165283]
chr4 [-4.297753810882568, 2.2052855491638184]
chr5 [-2.447938919067383, 1.7496592998504639]
chr6 [-5.278648376464844, 1.7017384767532349]
chr7 [-5.832656383514404, 2.78574800491333]
chr8 [-2.6853575706481934, 1.5217547416687012]
chr9 [-3.82442045211792, 3.2543280124664307]
chr10 [-3.6164724826812744, 3.1549313068389893]
chr11 [-1.8429272174835205, 1.5903886556625366]
chr12 [-1.6196963787078857, 1.5297129154205322]
chr13 [-1.8222864866256714, 1.3401376008987427]
chr14 [-1.4813015460968018, 1.5576770305633545]
chr15 [-6.29219388961792, 1.6035534143447876]
chr16 [-4.304950714111328, 1.465885877609253]
chr17 [-4.332077980041504, 2.3788046836853027]
chr18 [-1.570502519607544, 1.3708897829055786]
chr19 [-1.944730520248413, 1.2403700351715088]
chr20 [-1.6622285842895508, 1.3051719665527344]
chr21 [-3.746151924133301, 1.72748601436615]
chr22 [-3.993015289306

In [245]:
# IS value percentile
chr1_ = pd.Series(IS_dict['chr1'])
print(f'1% : {chr1_.quantile(0.01)}')
print(f'2% : {chr1_.quantile(0.02)}')
print(f'25% : {chr1_.quantile(0.25)}')
print(f'50% : {chr1_.quantile(0.50)}')
print(f'75% : {chr1_.quantile(0.75)}')
print(f'90% : {chr1_.quantile(0.90)}')
print(f'98% : {chr1_.quantile(0.98)}')
print(f'99% : {chr1_.quantile(0.99)}')

1% : -1.1851854956150054
2% : -0.9812185728549957
25% : -0.33465078473091125
50% : -0.07921354472637177
75% : 0.22643804550170898
90% : 0.5281458258628846
98% : 0.9428887200355538
99% : 1.1027782046794887


In [194]:
# calculate r by chr
for key in chr_bin_number.keys():
    print(f'Pearson r between IS and residual in {key} : ', round(stats.pearsonr(IS_dict[key], residual_dict[key])[0],2))

Pearson r between IS and residual in chr1 :  0.03
Pearson r between IS and residual in chr2 :  0.01
Pearson r between IS and residual in chr3 :  0.19
Pearson r between IS and residual in chr4 :  0.17
Pearson r between IS and residual in chr5 :  0.09
Pearson r between IS and residual in chr6 :  0.24
Pearson r between IS and residual in chr7 :  0.08
Pearson r between IS and residual in chr8 :  0.16
Pearson r between IS and residual in chr9 :  0.2
Pearson r between IS and residual in chr10 :  -0.03
Pearson r between IS and residual in chr11 :  0.22
Pearson r between IS and residual in chr12 :  0.09
Pearson r between IS and residual in chr13 :  0.04
Pearson r between IS and residual in chr14 :  0.22
Pearson r between IS and residual in chr15 :  0.31
Pearson r between IS and residual in chr16 :  0.03
Pearson r between IS and residual in chr17 :  0.1
Pearson r between IS and residual in chr18 :  0.18
Pearson r between IS and residual in chr19 :  0.01
Pearson r between IS and residual in chr2

In [195]:
# calculate r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between IS and residual in {key} : ', round(stats.spearmanr(IS_dict[key], residual_dict[key])[0],2))

Spearman r between IS and residual in chr1 :  0.25
Spearman r between IS and residual in chr2 :  0.05
Spearman r between IS and residual in chr3 :  0.21
Spearman r between IS and residual in chr4 :  0.23
Spearman r between IS and residual in chr5 :  0.13
Spearman r between IS and residual in chr6 :  0.32
Spearman r between IS and residual in chr7 :  0.12
Spearman r between IS and residual in chr8 :  0.23
Spearman r between IS and residual in chr9 :  0.3
Spearman r between IS and residual in chr10 :  0.04
Spearman r between IS and residual in chr11 :  0.32
Spearman r between IS and residual in chr12 :  0.14
Spearman r between IS and residual in chr13 :  0.21
Spearman r between IS and residual in chr14 :  0.28
Spearman r between IS and residual in chr15 :  0.28
Spearman r between IS and residual in chr16 :  0.21
Spearman r between IS and residual in chr17 :  0.09
Spearman r between IS and residual in chr18 :  0.18
Spearman r between IS and residual in chr19 :  0.28
Spearman r between IS 

In [196]:
residual_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    residual_flatten = residual_flatten + list(residual_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between IS and residual : ', round(stats.pearsonr(IS_flatten, residual_flatten)[0],2))
print(f'(Overall) Spearman r between IS and residual : ', round(stats.spearmanr(IS_flatten, residual_flatten)[0],2))

(Overall) Pearson r between IS and residual :  0.05
(Overall) Spearman r between IS and residual :  0.17


## r between normalized IS and residual

In [107]:
residual_dir = 'HiNTcnv_OUTPUT/K562_combined_dataForRegression/residualsPerChrom'

residual_dict = dict()
for file in os.listdir(residual_dir):
    with open(residual_dir + '/' + file, 'r') as f:
        chr_ = file.split('_')[1]
        residual_per_chr = []
        for line in f.readlines():
            residual_per_chr.append(float(line))
        residual_dict[chr_] = residual_per_chr

In [204]:
IS_path = 'FAN-C_Insulation/bed/GSE63525_K562_50k_IS_150kb.bed'

IS_dict = dict()
for key in chr_bin_number.keys():
    IS_dict[key] = np.zeros(chr_bin_number[key])

with open(IS_path, 'r') as f:
    for line in f.readlines():
        line_split = line.strip().split()
        chr_ = 'chr'+line_split[0]
        if chr_ in exclude_chr :
            continue
        bin_ = math.floor(int(line_split[1]) / resolution)
        if (line_split[4] == 'inf') or (line_split[4] == '-inf') or (line_split[4] == 'nan'):
            IS_dict[chr_][bin_] = 0
        else:
            IS_dict[chr_][bin_] = float(line_split[4])

In [205]:
# delete IS by zero lines

zero_line = dict()
for key in chr_bin_number.keys():
    zero_line[key] = list()
with open('exp2_IS_residual/zero_line.txt', 'r') as f :
    for line in f.readlines():
        line_split = line.strip().split()
        zero_line[line_split[0]].append(int(line_split[1]))

for key in chr_bin_number.keys():
    IS_dict[key] = np.delete(IS_dict[key], zero_line[key])

In [206]:
def scale_range (input, min, max):
    input += -(np.min(input))
    input /= np.max(input) / (max - min)
    input += min
    return input

In [207]:
# normalized to [0,1]
for key in chr_bin_number.keys():
    IS_dict[key] = scale_range(IS_dict[key], -1, 1)

In [208]:
# IS value interval
for key in chr_bin_number.keys():
    print(key, f'[{min(IS_dict[key])}, {max(IS_dict[key])}]')

chr1 [-1.0, 1.0]
chr2 [-1.0, 1.0]
chr3 [-1.0, 1.0]
chr4 [-1.0, 1.0]
chr5 [-1.0, 1.0]
chr6 [-1.0, 1.0]
chr7 [-1.0, 1.0]
chr8 [-1.0, 1.0]
chr9 [-1.0, 1.0]
chr10 [-1.0, 1.0]
chr11 [-1.0, 1.0]
chr12 [-1.0, 1.0]
chr13 [-1.0, 1.0]
chr14 [-1.0, 1.0]
chr15 [-1.0, 1.0]
chr16 [-1.0, 1.0]
chr17 [-1.0, 1.0]
chr18 [-1.0, 1.0]
chr19 [-1.0, 1.0]
chr20 [-1.0, 1.0]
chr21 [-1.0, 1.0]
chr22 [-1.0, 1.0]
chrX [-1.0, 1.0]


In [209]:
residual_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    residual_flatten = residual_flatten + list(residual_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between IS and residual : ', round(stats.pearsonr(IS_flatten, residual_flatten)[0],2))
print(f'(Overall) Spearman r between IS and residual : ', round(stats.spearmanr(IS_flatten, residual_flatten)[0],2))

(Overall) Pearson r between IS and residual :  0.12
(Overall) Spearman r between IS and residual :  0.35


In [210]:
# calculate r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between IS and residual in {key} : ', round(stats.spearmanr(IS_dict[key], residual_dict[key])[0],2))

Spearman r between IS and residual in chr1 :  0.25
Spearman r between IS and residual in chr2 :  0.05
Spearman r between IS and residual in chr3 :  0.21
Spearman r between IS and residual in chr4 :  0.23
Spearman r between IS and residual in chr5 :  0.13
Spearman r between IS and residual in chr6 :  0.32
Spearman r between IS and residual in chr7 :  0.12
Spearman r between IS and residual in chr8 :  0.23
Spearman r between IS and residual in chr9 :  0.3
Spearman r between IS and residual in chr10 :  0.04
Spearman r between IS and residual in chr11 :  0.32
Spearman r between IS and residual in chr12 :  0.14
Spearman r between IS and residual in chr13 :  0.21
Spearman r between IS and residual in chr14 :  0.28
Spearman r between IS and residual in chr15 :  0.28
Spearman r between IS and residual in chr16 :  0.21
Spearman r between IS and residual in chr17 :  0.09
Spearman r between IS and residual in chr18 :  0.18
Spearman r between IS and residual in chr19 :  0.28
Spearman r between IS 

In [211]:
# calculate r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between IS and residual in {key} : ', round(stats.spearmanr(IS_dict[key], residual_dict[key])[0],2))

Spearman r between IS and residual in chr1 :  0.25
Spearman r between IS and residual in chr2 :  0.05
Spearman r between IS and residual in chr3 :  0.21
Spearman r between IS and residual in chr4 :  0.23
Spearman r between IS and residual in chr5 :  0.13
Spearman r between IS and residual in chr6 :  0.32
Spearman r between IS and residual in chr7 :  0.12
Spearman r between IS and residual in chr8 :  0.23
Spearman r between IS and residual in chr9 :  0.3
Spearman r between IS and residual in chr10 :  0.04
Spearman r between IS and residual in chr11 :  0.32
Spearman r between IS and residual in chr12 :  0.14
Spearman r between IS and residual in chr13 :  0.21
Spearman r between IS and residual in chr14 :  0.28
Spearman r between IS and residual in chr15 :  0.28
Spearman r between IS and residual in chr16 :  0.21
Spearman r between IS and residual in chr17 :  0.09
Spearman r between IS and residual in chr18 :  0.18
Spearman r between IS and residual in chr19 :  0.28
Spearman r between IS 

## r between clipped IS and residual

In [246]:
IS_path = 'FAN-C_Insulation/bed/GSE63525_K562_50k_IS_150kb.bed'

IS_dict = dict()
for key in chr_bin_number.keys():
    IS_dict[key] = np.zeros(chr_bin_number[key])

with open(IS_path, 'r') as f:
    for line in f.readlines():
        line_split = line.strip().split()
        chr_ = 'chr'+line_split[0]
        if chr_ in exclude_chr :
            continue
        bin_ = math.floor(int(line_split[1]) / resolution)
        if (line_split[4] == 'inf') or (line_split[4] == '-inf') or (line_split[4] == 'nan'):
            IS_dict[chr_][bin_] = 0
        else:
            IS_dict[chr_][bin_] = float(line_split[4])


# delete IS by zero lines
zero_line = dict()
for key in chr_bin_number.keys():
    zero_line[key] = list()
with open('exp2_IS_residual/zero_line.txt', 'r') as f :
    for line in f.readlines():
        line_split = line.strip().split()
        zero_line[line_split[0]].append(int(line_split[1]))

for key in chr_bin_number.keys():
    IS_dict[key] = np.delete(IS_dict[key], zero_line[key])

In [247]:

for key in IS_dict.keys():
    IS_dict[key] = np.clip(IS_dict[key], -1, 1)
    print(key, f'[{min(IS_dict[key])}, {max(IS_dict[key])}]')

chr1 [-1.0, 1.0]
chr2 [-1.0, 1.0]
chr3 [-1.0, 1.0]
chr4 [-1.0, 1.0]
chr5 [-1.0, 1.0]
chr6 [-1.0, 1.0]
chr7 [-1.0, 1.0]
chr8 [-1.0, 1.0]
chr9 [-1.0, 1.0]
chr10 [-1.0, 1.0]
chr11 [-1.0, 1.0]
chr12 [-1.0, 1.0]
chr13 [-1.0, 1.0]
chr14 [-1.0, 1.0]
chr15 [-1.0, 1.0]
chr16 [-1.0, 1.0]
chr17 [-1.0, 1.0]
chr18 [-1.0, 1.0]
chr19 [-1.0, 1.0]
chr20 [-1.0, 1.0]
chr21 [-1.0, 1.0]
chr22 [-1.0, 1.0]
chrX [-1.0, 1.0]


In [248]:
residual_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    residual_flatten = residual_flatten + list(residual_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between clipped IS and residual : ', round(stats.pearsonr(IS_flatten, residual_flatten)[0],2))
print(f'(Overall) Spearman r between clipped IS and residual : ', round(stats.spearmanr(IS_flatten, residual_flatten)[0],2))

(Overall) Pearson r between clipped IS and residual :  0.06
(Overall) Spearman r between clipped IS and residual :  0.17


In [251]:
# calculate r by chr
for key in chr_bin_number.keys():
    print(f'Pearson r between clipped IS and residual in {key} : ', round(stats.pearsonr(IS_dict[key], residual_dict[key])[0],2))

Pearson r between clipped IS and residual in chr1 :  0.03
Pearson r between clipped IS and residual in chr2 :  0.03
Pearson r between clipped IS and residual in chr3 :  0.2
Pearson r between clipped IS and residual in chr4 :  0.19
Pearson r between clipped IS and residual in chr5 :  0.1
Pearson r between clipped IS and residual in chr6 :  0.26
Pearson r between clipped IS and residual in chr7 :  0.1
Pearson r between clipped IS and residual in chr8 :  0.16
Pearson r between clipped IS and residual in chr9 :  0.24
Pearson r between clipped IS and residual in chr10 :  -0.01
Pearson r between clipped IS and residual in chr11 :  0.22
Pearson r between clipped IS and residual in chr12 :  0.09
Pearson r between clipped IS and residual in chr13 :  0.06
Pearson r between clipped IS and residual in chr14 :  0.22
Pearson r between clipped IS and residual in chr15 :  0.29
Pearson r between clipped IS and residual in chr16 :  0.03
Pearson r between clipped IS and residual in chr17 :  0.07
Pearson 

In [250]:
# calculate r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between clipped IS and residual in {key} : ', round(stats.spearmanr(IS_dict[key], residual_dict[key])[0],2))

Spearman r between clipped IS and residual in chr1 :  0.25
Spearman r between clipped IS and residual in chr2 :  0.05
Spearman r between clipped IS and residual in chr3 :  0.21
Spearman r between clipped IS and residual in chr4 :  0.23
Spearman r between clipped IS and residual in chr5 :  0.13
Spearman r between clipped IS and residual in chr6 :  0.32
Spearman r between clipped IS and residual in chr7 :  0.12
Spearman r between clipped IS and residual in chr8 :  0.23
Spearman r between clipped IS and residual in chr9 :  0.3
Spearman r between clipped IS and residual in chr10 :  0.04
Spearman r between clipped IS and residual in chr11 :  0.32
Spearman r between clipped IS and residual in chr12 :  0.14
Spearman r between clipped IS and residual in chr13 :  0.21
Spearman r between clipped IS and residual in chr14 :  0.28
Spearman r between clipped IS and residual in chr15 :  0.28
Spearman r between clipped IS and residual in chr16 :  0.21
Spearman r between clipped IS and residual in chr1

## r between IS and bias
both include zero-line 

In [174]:
IS_path = 'FAN-C_Insulation/bed/GSE63525_K562_50k_IS_150kb.bed'

IS_dict = dict()
for key in chr_bin_number.keys():
    IS_dict[key] = np.zeros(chr_bin_number[key])

with open(IS_path, 'r') as f:
    for line in f.readlines():
        line_split = line.strip().split()
        chr_ = 'chr'+line_split[0]
        if chr_ in exclude_chr :
            continue
        bin_ = math.floor(int(line_split[1]) / resolution)
        if (line_split[4] == 'inf') or (line_split[4] == '-inf') or (line_split[4] == 'nan'):
            IS_dict[chr_][bin_] = 0
        else:
            IS_dict[chr_][bin_] = float(line_split[4])

#### load bias

In [148]:
# load ref genome 1kb GC
# but in 50kb way
GC_dict, counts = dict(), dict()
for key in chr_length.keys():
    GC_dict[key] = np.zeros(chr_bin_number[key])
    counts[key] = np.zeros(chr_bin_number[key])

with open(ref_genome_folder + '/hg19_1kb_GCPercent.bed', 'r') as f:
    for line in f.readlines()[1:]:
        line_split = line.strip().split()
        
        if line_split[0] in exclude_chr:
            continue
        else:
            start = math.floor(int(line_split[1]) / resolution)
            GC_dict[line_split[0]][start] += float(line_split[3])
            counts[line_split[0]][start] += 1

for key in GC_dict.keys():
    GC_dict[key] = GC_dict[key] / counts[key]

In [151]:
# this function is from HiNT github
import tabix
def getmappability(mappablity_track, matrixchrom, matrixstart, matrixend, resolution):
	tb = tabix.open(mappablity_track)
	records = tb.query(matrixchrom, int(matrixstart), int(matrixend))
	Allstart = int(matrixstart)
	Allend = int(matrixend)
	values = []
	binvalues = []
	values += [0]*(Allstart%resolution)
	for r in records:
		chrom,start,end,value = r
		length = int(end) - int(start)
		if int(start) == Allstart:
			temp = [float(value)]*length
		else:
			temp = [0]*(int(start)-Allstart) + [float(value)]*length
		values += temp
		Allstart = int(end)
	if Allstart == Allend:
		pass
	else:
		values += [0]*(Allend-Allstart)
	values += [0]*(resolution-Allend%resolution)
	tempcount = 0
	for i,v in enumerate(values):
		if i%resolution == (resolution-1):
			binvalues.append(float(tempcount)/float(resolution))
			tempcount = 0
		else:
			tempcount += v
	return binvalues

mappability_dict = dict()
for key in chr_length.keys() :
    mappability_dict[key] = np.array( getmappability(ref_genome_folder + '/hg19_mappability_50mer.bdg.gz', key, 1, chr_length[key], resolution) )

In [150]:
# load restriction sites 
num_res_sites_dict = dict()
for key in chr_length.keys() :
    num_res_sites_dict[key] = np.zeros(chr_bin_number[key])

with open(ref_genome_folder + '/hg19_HindIII_enzymeSites.txt', 'r') as f:
    for line in f.readlines():
        line_split = line.split()
        if line_split[0] in exclude_chr:
            continue
        else:
            for site in line_split[1:]:
                bin_ = math.floor( int(site) / resolution )
                num_res_sites_dict[line_split[0]][bin_] += 1

#### r between IS and GC

In [152]:
GC_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    GC_flatten = GC_flatten + list(GC_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between IS and GC : ', round(stats.pearsonr(IS_flatten, GC_flatten)[0],2))
print(f'(Overall) Spearman r between IS and GC : ', round(stats.spearmanr(IS_flatten, GC_flatten)[0],2))

(Overall) Pearson r between IS and GC :  0.22
(Overall) Spearman r between IS and GC :  0.35


In [153]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Pearson r between IS and GC in {key} : ', round(stats.pearsonr(IS_dict[key], GC_dict[key])[0],2))

Pearson r between IS and GC in chr1 :  0.24
Pearson r between IS and GC in chr2 :  0.15
Pearson r between IS and GC in chr3 :  0.24
Pearson r between IS and GC in chr4 :  0.19
Pearson r between IS and GC in chr5 :  0.21
Pearson r between IS and GC in chr6 :  0.21
Pearson r between IS and GC in chr7 :  0.18
Pearson r between IS and GC in chr8 :  0.15
Pearson r between IS and GC in chr9 :  0.26
Pearson r between IS and GC in chr10 :  0.18
Pearson r between IS and GC in chr11 :  0.24
Pearson r between IS and GC in chr12 :  0.22
Pearson r between IS and GC in chr13 :  0.16
Pearson r between IS and GC in chr14 :  0.32
Pearson r between IS and GC in chr15 :  0.29
Pearson r between IS and GC in chr16 :  0.25
Pearson r between IS and GC in chr17 :  0.2
Pearson r between IS and GC in chr18 :  0.1
Pearson r between IS and GC in chr19 :  0.25
Pearson r between IS and GC in chr20 :  0.16
Pearson r between IS and GC in chr21 :  0.46
Pearson r between IS and GC in chr22 :  0.58
Pearson r between IS 

In [154]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between IS and GC in {key} : ', round(stats.spearmanr(IS_dict[key], GC_dict[key])[0],2))

Spearman r between IS and GC in chr1 :  0.4
Spearman r between IS and GC in chr2 :  0.29
Spearman r between IS and GC in chr3 :  0.37
Spearman r between IS and GC in chr4 :  0.37
Spearman r between IS and GC in chr5 :  0.35
Spearman r between IS and GC in chr6 :  0.35
Spearman r between IS and GC in chr7 :  0.26
Spearman r between IS and GC in chr8 :  0.27
Spearman r between IS and GC in chr9 :  0.4
Spearman r between IS and GC in chr10 :  0.28
Spearman r between IS and GC in chr11 :  0.39
Spearman r between IS and GC in chr12 :  0.4
Spearman r between IS and GC in chr13 :  0.39
Spearman r between IS and GC in chr14 :  0.49
Spearman r between IS and GC in chr15 :  0.35
Spearman r between IS and GC in chr16 :  0.42
Spearman r between IS and GC in chr17 :  0.28
Spearman r between IS and GC in chr18 :  0.31
Spearman r between IS and GC in chr19 :  0.3
Spearman r between IS and GC in chr20 :  0.29
Spearman r between IS and GC in chr21 :  0.57
Spearman r between IS and GC in chr22 :  0.56
S

#### r between IS and mappability

In [175]:
mappability_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    mappability_flatten = mappability_flatten + list(mappability_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between IS and mappability : ', round(stats.pearsonr(IS_flatten, mappability_flatten)[0],2))
print(f'(Overall) Spearman r between IS and mappability : ', round(stats.spearmanr(IS_flatten, mappability_flatten)[0],2))

(Overall) Pearson r between IS and mappability :  0.09
(Overall) Spearman r between IS and mappability :  -0.03


In [177]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Pearson r between IS and mappability in {key} : ', round(stats.pearsonr(IS_dict[key], mappability_dict[key])[0],2))

Pearson r between IS and mappability in chr1 :  0.11
Pearson r between IS and mappability in chr2 :  -0.04
Pearson r between IS and mappability in chr3 :  -0.1
Pearson r between IS and mappability in chr4 :  -0.14
Pearson r between IS and mappability in chr5 :  -0.03
Pearson r between IS and mappability in chr6 :  -0.02
Pearson r between IS and mappability in chr7 :  -0.02
Pearson r between IS and mappability in chr8 :  0.07
Pearson r between IS and mappability in chr9 :  0.28
Pearson r between IS and mappability in chr10 :  0.08
Pearson r between IS and mappability in chr11 :  -0.06
Pearson r between IS and mappability in chr12 :  -0.12
Pearson r between IS and mappability in chr13 :  0.02
Pearson r between IS and mappability in chr14 :  0.19
Pearson r between IS and mappability in chr15 :  0.34
Pearson r between IS and mappability in chr16 :  0.18
Pearson r between IS and mappability in chr17 :  0.13
Pearson r between IS and mappability in chr18 :  -0.11
Pearson r between IS and mapp

In [178]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between IS and mappability in {key} : ', round(stats.spearmanr(IS_dict[key], mappability_dict[key])[0],2))

Spearman r between IS and mappability in chr1 :  -0.04
Spearman r between IS and mappability in chr2 :  -0.14
Spearman r between IS and mappability in chr3 :  -0.14
Spearman r between IS and mappability in chr4 :  -0.17
Spearman r between IS and mappability in chr5 :  -0.09
Spearman r between IS and mappability in chr6 :  -0.08
Spearman r between IS and mappability in chr7 :  -0.14
Spearman r between IS and mappability in chr8 :  -0.04
Spearman r between IS and mappability in chr9 :  0.23
Spearman r between IS and mappability in chr10 :  -0.06
Spearman r between IS and mappability in chr11 :  -0.1
Spearman r between IS and mappability in chr12 :  -0.17
Spearman r between IS and mappability in chr13 :  -0.11
Spearman r between IS and mappability in chr14 :  0.1
Spearman r between IS and mappability in chr15 :  0.31
Spearman r between IS and mappability in chr16 :  -0.01
Spearman r between IS and mappability in chr17 :  -0.03
Spearman r between IS and mappability in chr18 :  -0.2
Spearma

#### r between IS and number of res. site

In [159]:
num_res_sites_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    num_res_sites_flatten = num_res_sites_flatten + list(num_res_sites_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between IS and num_res_sites : ', round(stats.pearsonr(IS_flatten, num_res_sites_flatten)[0],2))
print(f'(Overall) Spearman r between IS and num_res_sites : ', round(stats.spearmanr(IS_flatten, num_res_sites_flatten)[0],2))

(Overall) Pearson r between IS and num_res_sites :  -0.06
(Overall) Spearman r between IS and num_res_sites :  -0.09


In [160]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Pearson r between IS and num_res_sites in {key} : ', round(stats.pearsonr(IS_dict[key], num_res_sites_dict[key])[0],2))

Pearson r between IS and num_res_sites in chr1 :  -0.05
Pearson r between IS and num_res_sites in chr2 :  -0.15
Pearson r between IS and num_res_sites in chr3 :  -0.19
Pearson r between IS and num_res_sites in chr4 :  -0.14
Pearson r between IS and num_res_sites in chr5 :  -0.08
Pearson r between IS and num_res_sites in chr6 :  -0.13
Pearson r between IS and num_res_sites in chr7 :  -0.13
Pearson r between IS and num_res_sites in chr8 :  -0.16
Pearson r between IS and num_res_sites in chr9 :  0.05
Pearson r between IS and num_res_sites in chr10 :  -0.09
Pearson r between IS and num_res_sites in chr11 :  -0.17
Pearson r between IS and num_res_sites in chr12 :  -0.15
Pearson r between IS and num_res_sites in chr13 :  -0.02
Pearson r between IS and num_res_sites in chr14 :  0.03
Pearson r between IS and num_res_sites in chr15 :  0.2
Pearson r between IS and num_res_sites in chr16 :  -0.03
Pearson r between IS and num_res_sites in chr17 :  -0.08
Pearson r between IS and num_res_sites in ch

In [161]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between IS and num_res_sites in {key} : ', round(stats.spearmanr(IS_dict[key], num_res_sites_dict[key])[0],2))

Spearman r between IS and num_res_sites in chr1 :  -0.11
Spearman r between IS and num_res_sites in chr2 :  -0.17
Spearman r between IS and num_res_sites in chr3 :  -0.18
Spearman r between IS and num_res_sites in chr4 :  -0.12
Spearman r between IS and num_res_sites in chr5 :  -0.09
Spearman r between IS and num_res_sites in chr6 :  -0.14
Spearman r between IS and num_res_sites in chr7 :  -0.17
Spearman r between IS and num_res_sites in chr8 :  -0.16
Spearman r between IS and num_res_sites in chr9 :  0.08
Spearman r between IS and num_res_sites in chr10 :  -0.13
Spearman r between IS and num_res_sites in chr11 :  -0.23
Spearman r between IS and num_res_sites in chr12 :  -0.21
Spearman r between IS and num_res_sites in chr13 :  -0.03
Spearman r between IS and num_res_sites in chr14 :  -0.01
Spearman r between IS and num_res_sites in chr15 :  0.28
Spearman r between IS and num_res_sites in chr16 :  -0.04
Spearman r between IS and num_res_sites in chr17 :  -0.11
Spearman r between IS and

## r between normalizaed IS and bias
both include zero-line 

In [162]:
def scale_range (input, min, max):
    input += -(np.min(input))
    input /= np.max(input) / (max - min)
    input += min
    return input

In [163]:
# normalized to [0,1]
for key in chr_bin_number.keys():
    IS_dict[key] = scale_range(IS_dict[key], -1, 1)

#### r between normalized IS and GC

In [164]:
GC_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    GC_flatten = GC_flatten + list(GC_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between normalized IS and GC : ', round(stats.pearsonr(IS_flatten, GC_flatten)[0],2))
print(f'(Overall) Spearman r between normalized IS and GC : ', round(stats.spearmanr(IS_flatten, GC_flatten)[0],2))

(Overall) Pearson r between normalized IS and GC :  0.13
(Overall) Spearman r between normalized IS and GC :  0.25


In [166]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Pearson r between normalized IS and GC in {key} : ', round(stats.pearsonr(IS_dict[key], GC_dict[key])[0],2))

Pearson r between normalized IS and GC in chr1 :  0.24
Pearson r between normalized IS and GC in chr2 :  0.15
Pearson r between normalized IS and GC in chr3 :  0.24
Pearson r between normalized IS and GC in chr4 :  0.19
Pearson r between normalized IS and GC in chr5 :  0.21
Pearson r between normalized IS and GC in chr6 :  0.21
Pearson r between normalized IS and GC in chr7 :  0.18
Pearson r between normalized IS and GC in chr8 :  0.15
Pearson r between normalized IS and GC in chr9 :  0.26
Pearson r between normalized IS and GC in chr10 :  0.18
Pearson r between normalized IS and GC in chr11 :  0.24
Pearson r between normalized IS and GC in chr12 :  0.22
Pearson r between normalized IS and GC in chr13 :  0.16
Pearson r between normalized IS and GC in chr14 :  0.32
Pearson r between normalized IS and GC in chr15 :  0.29
Pearson r between normalized IS and GC in chr16 :  0.25
Pearson r between normalized IS and GC in chr17 :  0.2
Pearson r between normalized IS and GC in chr18 :  0.1
Pea

In [167]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between normalized IS and GC in {key} : ', round(stats.spearmanr(IS_dict[key], GC_dict[key])[0],2))

Spearman r between normalized IS and GC in chr1 :  0.4
Spearman r between normalized IS and GC in chr2 :  0.29
Spearman r between normalized IS and GC in chr3 :  0.37
Spearman r between normalized IS and GC in chr4 :  0.37
Spearman r between normalized IS and GC in chr5 :  0.35
Spearman r between normalized IS and GC in chr6 :  0.35
Spearman r between normalized IS and GC in chr7 :  0.26
Spearman r between normalized IS and GC in chr8 :  0.27
Spearman r between normalized IS and GC in chr9 :  0.4
Spearman r between normalized IS and GC in chr10 :  0.28
Spearman r between normalized IS and GC in chr11 :  0.39
Spearman r between normalized IS and GC in chr12 :  0.4
Spearman r between normalized IS and GC in chr13 :  0.39
Spearman r between normalized IS and GC in chr14 :  0.49
Spearman r between normalized IS and GC in chr15 :  0.35
Spearman r between normalized IS and GC in chr16 :  0.42
Spearman r between normalized IS and GC in chr17 :  0.28
Spearman r between normalized IS and GC in 

#### r between normalized IS and mappability

In [168]:
mappability_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    mappability_flatten = mappability_flatten + list(mappability_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between normalized IS and mappability : ', round(stats.pearsonr(IS_flatten, mappability_flatten)[0],2))
print(f'(Overall) Spearman r between normalized IS and mappability : ', round(stats.spearmanr(IS_flatten, mappability_flatten)[0],2))

(Overall) Pearson r between normalized IS and mappability :  -0.0
(Overall) Spearman r between normalized IS and mappability :  -0.06


In [169]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Pearson r between normalized IS and mappability in {key} : ', round(stats.pearsonr(IS_dict[key], mappability_dict[key])[0],2))

Pearson r between normalized IS and mappability in chr1 :  0.11
Pearson r between normalized IS and mappability in chr2 :  -0.04
Pearson r between normalized IS and mappability in chr3 :  -0.1
Pearson r between normalized IS and mappability in chr4 :  -0.14
Pearson r between normalized IS and mappability in chr5 :  -0.03
Pearson r between normalized IS and mappability in chr6 :  -0.02
Pearson r between normalized IS and mappability in chr7 :  -0.02
Pearson r between normalized IS and mappability in chr8 :  0.07
Pearson r between normalized IS and mappability in chr9 :  0.28
Pearson r between normalized IS and mappability in chr10 :  0.08
Pearson r between normalized IS and mappability in chr11 :  -0.06
Pearson r between normalized IS and mappability in chr12 :  -0.12
Pearson r between normalized IS and mappability in chr13 :  0.02
Pearson r between normalized IS and mappability in chr14 :  0.19
Pearson r between normalized IS and mappability in chr15 :  0.34
Pearson r between normalize

In [170]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between normalized IS and mappability in {key} : ', round(stats.spearmanr(IS_dict[key], mappability_dict[key])[0],2))

Spearman r between normalized IS and mappability in chr1 :  -0.04
Spearman r between normalized IS and mappability in chr2 :  -0.14
Spearman r between normalized IS and mappability in chr3 :  -0.14
Spearman r between normalized IS and mappability in chr4 :  -0.17
Spearman r between normalized IS and mappability in chr5 :  -0.09
Spearman r between normalized IS and mappability in chr6 :  -0.08
Spearman r between normalized IS and mappability in chr7 :  -0.14
Spearman r between normalized IS and mappability in chr8 :  -0.04
Spearman r between normalized IS and mappability in chr9 :  0.23
Spearman r between normalized IS and mappability in chr10 :  -0.06
Spearman r between normalized IS and mappability in chr11 :  -0.1
Spearman r between normalized IS and mappability in chr12 :  -0.17
Spearman r between normalized IS and mappability in chr13 :  -0.11
Spearman r between normalized IS and mappability in chr14 :  0.1
Spearman r between normalized IS and mappability in chr15 :  0.31
Spearman 

#### r between normalized IS and number of res. site

In [171]:
num_res_sites_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    num_res_sites_flatten = num_res_sites_flatten + list(num_res_sites_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between normalized IS and num_res_sites : ', round(stats.pearsonr(IS_flatten, num_res_sites_flatten)[0],2))
print(f'(Overall) Spearman r between normalized IS and num_res_sites : ', round(stats.spearmanr(IS_flatten, num_res_sites_flatten)[0],2))

(Overall) Pearson r between normalized IS and num_res_sites :  -0.1
(Overall) Spearman r between normalized IS and num_res_sites :  -0.12


In [172]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Pearson r between normalized IS and num_res_sites in {key} : ', round(stats.pearsonr(IS_dict[key], num_res_sites_dict[key])[0],2))

Pearson r between normalized IS and num_res_sites in chr1 :  -0.05
Pearson r between normalized IS and num_res_sites in chr2 :  -0.15
Pearson r between normalized IS and num_res_sites in chr3 :  -0.19
Pearson r between normalized IS and num_res_sites in chr4 :  -0.14
Pearson r between normalized IS and num_res_sites in chr5 :  -0.08
Pearson r between normalized IS and num_res_sites in chr6 :  -0.13
Pearson r between normalized IS and num_res_sites in chr7 :  -0.13
Pearson r between normalized IS and num_res_sites in chr8 :  -0.16
Pearson r between normalized IS and num_res_sites in chr9 :  0.05
Pearson r between normalized IS and num_res_sites in chr10 :  -0.09
Pearson r between normalized IS and num_res_sites in chr11 :  -0.17
Pearson r between normalized IS and num_res_sites in chr12 :  -0.15
Pearson r between normalized IS and num_res_sites in chr13 :  -0.02
Pearson r between normalized IS and num_res_sites in chr14 :  0.03
Pearson r between normalized IS and num_res_sites in chr15 

In [173]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between normalized IS and num_res_sites in {key} : ', round(stats.spearmanr(IS_dict[key], num_res_sites_dict[key])[0],2))

Spearman r between normalized IS and num_res_sites in chr1 :  -0.11
Spearman r between normalized IS and num_res_sites in chr2 :  -0.17
Spearman r between normalized IS and num_res_sites in chr3 :  -0.18
Spearman r between normalized IS and num_res_sites in chr4 :  -0.12
Spearman r between normalized IS and num_res_sites in chr5 :  -0.09
Spearman r between normalized IS and num_res_sites in chr6 :  -0.14
Spearman r between normalized IS and num_res_sites in chr7 :  -0.17
Spearman r between normalized IS and num_res_sites in chr8 :  -0.16
Spearman r between normalized IS and num_res_sites in chr9 :  0.08
Spearman r between normalized IS and num_res_sites in chr10 :  -0.13
Spearman r between normalized IS and num_res_sites in chr11 :  -0.23
Spearman r between normalized IS and num_res_sites in chr12 :  -0.21
Spearman r between normalized IS and num_res_sites in chr13 :  -0.03
Spearman r between normalized IS and num_res_sites in chr14 :  -0.01
Spearman r between normalized IS and num_res

## r between clipped IS and bias
both include zero-line 

In [255]:
# load IS
IS_path = 'FAN-C_Insulation/bed/GSE63525_K562_50k_IS_150kb.bed'

IS_dict = dict()
for key in chr_bin_number.keys():
    IS_dict[key] = np.zeros(chr_bin_number[key])

with open(IS_path, 'r') as f:
    for line in f.readlines():
        line_split = line.strip().split()
        chr_ = 'chr'+line_split[0]
        if chr_ in exclude_chr :
            continue
        bin_ = math.floor(int(line_split[1]) / resolution)
        if (line_split[4] == 'inf') or (line_split[4] == '-inf') or (line_split[4] == 'nan'):
            IS_dict[chr_][bin_] = 0
        else:
            IS_dict[chr_][bin_] = float(line_split[4])


# clip
for key in IS_dict.keys():
    IS_dict[key] = np.clip(IS_dict[key], -1, 1)

#### r between clipped IS and GC

In [256]:
GC_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    GC_flatten = GC_flatten + list(GC_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between clipped IS and GC : ', round(stats.pearsonr(IS_flatten, GC_flatten)[0],2))
print(f'(Overall) Spearman r between clipped IS and GC : ', round(stats.spearmanr(IS_flatten, GC_flatten)[0],2))

(Overall) Pearson r between normalized IS and GC :  0.23
(Overall) Spearman r between normalized IS and GC :  0.35


In [257]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Pearson r between clipped IS and GC in {key} : ', round(stats.pearsonr(IS_dict[key], GC_dict[key])[0],2))

Pearson r between clipped IS and GC in chr1 :  0.25
Pearson r between clipped IS and GC in chr2 :  0.16
Pearson r between clipped IS and GC in chr3 :  0.24
Pearson r between clipped IS and GC in chr4 :  0.2
Pearson r between clipped IS and GC in chr5 :  0.22
Pearson r between clipped IS and GC in chr6 :  0.2
Pearson r between clipped IS and GC in chr7 :  0.18
Pearson r between clipped IS and GC in chr8 :  0.16
Pearson r between clipped IS and GC in chr9 :  0.31
Pearson r between clipped IS and GC in chr10 :  0.18
Pearson r between clipped IS and GC in chr11 :  0.25
Pearson r between clipped IS and GC in chr12 :  0.23
Pearson r between clipped IS and GC in chr13 :  0.17
Pearson r between clipped IS and GC in chr14 :  0.33
Pearson r between clipped IS and GC in chr15 :  0.34
Pearson r between clipped IS and GC in chr16 :  0.28
Pearson r between clipped IS and GC in chr17 :  0.22
Pearson r between clipped IS and GC in chr18 :  0.1
Pearson r between clipped IS and GC in chr19 :  0.25
Pears

In [258]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between clipped IS and GC in {key} : ', round(stats.spearmanr(IS_dict[key], GC_dict[key])[0],2))

Spearman r between clipped IS and GC in chr1 :  0.4
Spearman r between clipped IS and GC in chr2 :  0.29
Spearman r between clipped IS and GC in chr3 :  0.37
Spearman r between clipped IS and GC in chr4 :  0.37
Spearman r between clipped IS and GC in chr5 :  0.35
Spearman r between clipped IS and GC in chr6 :  0.35
Spearman r between clipped IS and GC in chr7 :  0.26
Spearman r between clipped IS and GC in chr8 :  0.27
Spearman r between clipped IS and GC in chr9 :  0.4
Spearman r between clipped IS and GC in chr10 :  0.28
Spearman r between clipped IS and GC in chr11 :  0.39
Spearman r between clipped IS and GC in chr12 :  0.4
Spearman r between clipped IS and GC in chr13 :  0.39
Spearman r between clipped IS and GC in chr14 :  0.49
Spearman r between clipped IS and GC in chr15 :  0.35
Spearman r between clipped IS and GC in chr16 :  0.42
Spearman r between clipped IS and GC in chr17 :  0.28
Spearman r between clipped IS and GC in chr18 :  0.31
Spearman r between clipped IS and GC in 

#### r between clipped IS and mappability

In [259]:
mappability_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    mappability_flatten = mappability_flatten + list(mappability_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between clipped IS and mappability : ', round(stats.pearsonr(IS_flatten, mappability_flatten)[0],2))
print(f'(Overall) Spearman r between clipped IS and mappability : ', round(stats.spearmanr(IS_flatten, mappability_flatten)[0],2))

(Overall) Pearson r between clipped IS and mappability :  0.08
(Overall) Spearman r between clipped IS and mappability :  -0.03


In [260]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Pearson r between clipped IS and mappability in {key} : ', round(stats.pearsonr(IS_dict[key], mappability_dict[key])[0],2))

Pearson r between clipped IS and mappability in chr1 :  0.1
Pearson r between clipped IS and mappability in chr2 :  -0.05
Pearson r between clipped IS and mappability in chr3 :  -0.1
Pearson r between clipped IS and mappability in chr4 :  -0.15
Pearson r between clipped IS and mappability in chr5 :  -0.04
Pearson r between clipped IS and mappability in chr6 :  -0.05
Pearson r between clipped IS and mappability in chr7 :  -0.04
Pearson r between clipped IS and mappability in chr8 :  0.03
Pearson r between clipped IS and mappability in chr9 :  0.29
Pearson r between clipped IS and mappability in chr10 :  0.05
Pearson r between clipped IS and mappability in chr11 :  -0.06
Pearson r between clipped IS and mappability in chr12 :  -0.12
Pearson r between clipped IS and mappability in chr13 :  0.03
Pearson r between clipped IS and mappability in chr14 :  0.19
Pearson r between clipped IS and mappability in chr15 :  0.36
Pearson r between clipped IS and mappability in chr16 :  0.17
Pearson r b

In [261]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between clipped IS and mappability in {key} : ', round(stats.spearmanr(IS_dict[key], mappability_dict[key])[0],2))

Spearman r between clipped IS and mappability in chr1 :  -0.04
Spearman r between clipped IS and mappability in chr2 :  -0.14
Spearman r between clipped IS and mappability in chr3 :  -0.14
Spearman r between clipped IS and mappability in chr4 :  -0.17
Spearman r between clipped IS and mappability in chr5 :  -0.09
Spearman r between clipped IS and mappability in chr6 :  -0.08
Spearman r between clipped IS and mappability in chr7 :  -0.14
Spearman r between clipped IS and mappability in chr8 :  -0.04
Spearman r between clipped IS and mappability in chr9 :  0.23
Spearman r between clipped IS and mappability in chr10 :  -0.06
Spearman r between clipped IS and mappability in chr11 :  -0.1
Spearman r between clipped IS and mappability in chr12 :  -0.17
Spearman r between clipped IS and mappability in chr13 :  -0.11
Spearman r between clipped IS and mappability in chr14 :  0.1
Spearman r between clipped IS and mappability in chr15 :  0.31
Spearman r between clipped IS and mappability in chr16

#### r between clipped IS and number of res. site

In [262]:
num_res_sites_flatten, IS_flatten = [], []
for key in chr_bin_number.keys():
    num_res_sites_flatten = num_res_sites_flatten + list(num_res_sites_dict[key])
    IS_flatten = IS_flatten + list(IS_dict[key])

print(f'(Overall) Pearson r between clipped IS and num_res_sites : ', round(stats.pearsonr(IS_flatten, num_res_sites_flatten)[0],2))
print(f'(Overall) Spearman r between clipped IS and num_res_sites : ', round(stats.spearmanr(IS_flatten, num_res_sites_flatten)[0],2))

(Overall) Pearson r between clipped IS and num_res_sites :  -0.06
(Overall) Spearman r between clipped IS and num_res_sites :  -0.09


In [263]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Pearson r between clipped IS and num_res_sites in {key} : ', round(stats.pearsonr(IS_dict[key], num_res_sites_dict[key])[0],2))

Pearson r between clipped IS and num_res_sites in chr1 :  -0.06
Pearson r between clipped IS and num_res_sites in chr2 :  -0.15
Pearson r between clipped IS and num_res_sites in chr3 :  -0.19
Pearson r between clipped IS and num_res_sites in chr4 :  -0.15
Pearson r between clipped IS and num_res_sites in chr5 :  -0.09
Pearson r between clipped IS and num_res_sites in chr6 :  -0.14
Pearson r between clipped IS and num_res_sites in chr7 :  -0.15
Pearson r between clipped IS and num_res_sites in chr8 :  -0.17
Pearson r between clipped IS and num_res_sites in chr9 :  0.06
Pearson r between clipped IS and num_res_sites in chr10 :  -0.12
Pearson r between clipped IS and num_res_sites in chr11 :  -0.17
Pearson r between clipped IS and num_res_sites in chr12 :  -0.15
Pearson r between clipped IS and num_res_sites in chr13 :  -0.01
Pearson r between clipped IS and num_res_sites in chr14 :  0.03
Pearson r between clipped IS and num_res_sites in chr15 :  0.23
Pearson r between clipped IS and num_

In [264]:
# r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between clipped IS and num_res_sites in {key} : ', round(stats.spearmanr(IS_dict[key], num_res_sites_dict[key])[0],2))

Spearman r between clipped IS and num_res_sites in chr1 :  -0.11
Spearman r between clipped IS and num_res_sites in chr2 :  -0.17
Spearman r between clipped IS and num_res_sites in chr3 :  -0.18
Spearman r between clipped IS and num_res_sites in chr4 :  -0.12
Spearman r between clipped IS and num_res_sites in chr5 :  -0.09
Spearman r between clipped IS and num_res_sites in chr6 :  -0.14
Spearman r between clipped IS and num_res_sites in chr7 :  -0.17
Spearman r between clipped IS and num_res_sites in chr8 :  -0.16
Spearman r between clipped IS and num_res_sites in chr9 :  0.08
Spearman r between clipped IS and num_res_sites in chr10 :  -0.13
Spearman r between clipped IS and num_res_sites in chr11 :  -0.23
Spearman r between clipped IS and num_res_sites in chr12 :  -0.21
Spearman r between clipped IS and num_res_sites in chr13 :  -0.03
Spearman r between clipped IS and num_res_sites in chr14 :  -0.01
Spearman r between clipped IS and num_res_sites in chr15 :  0.28
Spearman r between cl

## Output IS for BICseq input

In [240]:
base_dir = 'exp2_IS_residual'
dir_ = 'output_clip_IS_for_BICseq/'
file_prefix = 'K562_clip_IS'
if not os.path.isdir(f'{base_dir}/{dir_}'): os.mkdir(f'{base_dir}/{dir_}')

config_file = open(f'{base_dir}/{dir_}/{file_prefix}_config.txt', 'w+')
config_file.write(f'chr\tcase\n')

for key in chr_bin_number.keys():
    config_file.write(f'{key}\t{dir_}/{file_prefix}_{key}.txt\n')
    f = open(f'{base_dir}/{dir_}/{file_prefix}_{key}.txt', 'w+')
    f.write('<Start Position of the Bin>\t<End Position of the Bin>\t<Observed Read count in the Bin>\t<Expected Read Count in the Bin>\n')

    expected = min(IS_dict[key])
    for i, v in enumerate(IS_dict[key]):
        bin_start = i * resolution
        bin_end = (i+1) * resolution -1 if ((i+1) * resolution -1) < chr_length[key] else chr_length[key]
        observed = v - expected
        f.write(f'{bin_start}\t{bin_end}\t{observed}\t{0-expected}\n')
    f.close()
config_file.close()


### IS with WGS CNV 

In [9]:
IS_path = 'Insulation_score/GSE63525_K562_50k_IS_150kb.bed'

IS_dict = dict()
for key in chr_bin_number.keys():
    IS_dict[key] = np.zeros(chr_bin_number[key])

with open(IS_path, 'r') as f:
    for line in f.readlines():
        line_split = line.strip().split()
        chr_ = 'chr'+line_split[0]
        if chr_ in exclude_chr :
            continue
        bin_ = math.floor(int(line_split[1]) / resolution)
        if (line_split[4] == 'inf') or (line_split[4] == '-inf') or (line_split[4] == 'nan'):
            IS_dict[chr_][bin_] = 0
        else:
            IS_dict[chr_][bin_] = float(line_split[4])

In [5]:
wgs_cnv_file = f'K562_WGS_CNV/K562_WGS_CNV_lambda4.txt'
wgs_cnv = load_seg_file(wgs_cnv_file, has_p_value = False)

In [10]:
# calculate r by chr
for key in chr_bin_number.keys():
    print(f'Spearman r between residual and WGS segments in {key} : ', round(stats.spearmanr(IS_dict[key], wgs_cnv[key])[0],2))

Spearman r between residual and WGS segments in chr1 :  0.25
Spearman r between residual and WGS segments in chr2 :  -0.22
Spearman r between residual and WGS segments in chr3 :  0.03
Spearman r between residual and WGS segments in chr4 :  0.0
Spearman r between residual and WGS segments in chr5 :  -0.05
Spearman r between residual and WGS segments in chr6 :  0.35
Spearman r between residual and WGS segments in chr7 :  -0.04
Spearman r between residual and WGS segments in chr8 :  nan
Spearman r between residual and WGS segments in chr9 :  0.11
Spearman r between residual and WGS segments in chr10 :  -0.18
Spearman r between residual and WGS segments in chr11 :  0.29
Spearman r between residual and WGS segments in chr12 :  -0.15
Spearman r between residual and WGS segments in chr13 :  0.12
Spearman r between residual and WGS segments in chr14 :  -0.15
Spearman r between residual and WGS segments in chr15 :  0.48
Spearman r between residual and WGS segments in chr16 :  0.22
Spearman r be

/Users/weihanc./opt/anaconda3/envs/hic/lib/python3.7/site-packages/scipy/stats/stats.py:4484: SpearmanRConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(SpearmanRConstantInputWarning())
